In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import utils
from utils import sizes, prices, distances, mean_squared_error, plot_1d

## Polynomial Regression
### Motivation
For *simple regression* $x, 1\in\mathbb{R}^{m}$: Instead of solving $\underset{w}{\mathrm{arg\,min}}\; \lVert w_1x + w_0 1 - y\rVert^2_2$ i.e.
$$
\underset{w}{\mathrm{arg\,min}}\; \lVert \sum^1_{k=0}w_kx^k - y\rVert^2_2
$$
where $\cdot^k$ is applied element-wise, we extend the model class to polynomials of degree $d$ and solve
$$
\underset{w}{\mathrm{arg\,min}}\; \lVert \sum^d_{k=0}w_kx^k - y\rVert^2_2
$$

Note:
- this can be extended to the multivariate case

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact, fixed

# Function to plot with adjustable slope and intercept
def update_plot(ax, polynomial, quadratic=0, slope=1, intercept=0):
    title = "Interactive Linear Regression"
    legend = lambda _, b, c: f"Line: y = {b:.2f}x + {c:.2f}"
    if polynomial:  # used further below
        title = "Interactive Polynomial Regression"
        legend = lambda a, b, c: f"Line: y = {a:.2f}x^2 + {b:.2f}x + {c:.2f}"

    # Clear the axes but keep the figure
    ax.clear()
    
    # Scatter plot of the actual data
    ax.scatter(sizes, prices, color="C0", label='Actual Prices')
    
    # Line equation: y = slope * x + intercept
    predicted_prices = quadratic * sizes ** 2 + slope * sizes + intercept
    
    # Plot the fitted line with the current quadratic, slope and intercept
    sizes_grid = np.linspace(sizes.min(), sizes.max(), 50)
    predicted_prices_grid = quadratic * sizes_grid ** 2 + slope * sizes_grid + intercept
    ax.plot(sizes_grid, predicted_prices_grid, color="C1", label=legend(quadratic, slope, intercept), linestyle='--')

    # Draw error bars: vertical lines between actual and predicted values - more efficient approach
    error_lines = []
    for i in range(len(sizes)):
        error_lines.extend([[sizes[i], sizes[i]], [prices[i], predicted_prices[i]], [None, None]])
    
    # Plot all error lines at once (more efficient than individual plots)
    x_coords = [error_lines[i] for i in range(0, len(error_lines), 3)]
    y_coords = [error_lines[i] for i in range(1, len(error_lines), 3)]
    
    for x_line, y_line in zip(x_coords, y_coords):
        ax.plot(x_line, y_line, color="C2", linestyle='-', lw=1.5)

    # Calculate and display the mean squared error (MSE)
    mse = mean_squared_error(prices, predicted_prices, verbose=False)
    ax.text(0.5, 0.9, f'Mean Squared Error: {mse:,.1f}', 
            horizontalalignment='center', verticalalignment='center', 
            transform=ax.transAxes, fontsize=12, color='black',
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

    # Labels, title, and legend
    ax.set_title(title)
    ax.set_xlabel('Size of Apartment (square meters)')
    ax.set_ylabel('Price (EUR)')
    ax.legend()
    ax.grid(True)
    
    # Redraw the canvas
    fig.canvas.draw_idle()

%matplotlib ipympl
plt.close()

# Create persistent figure and axes for better performance
fig, ax = plt.subplots(figsize=(10, 6))

# Interactive widgets for slope and intercept
interact(update_plot, ax=fixed(ax), polynomial=fixed(True),
         quadratic=widgets.FloatSlider(min=-1, max=1, step=0.01, value=0),
         slope=widgets.FloatSlider(min=-80, max=80, step=0.1, value=1), 
         intercept=widgets.FloatSlider(min=-1000, max=3000, step=50, value=0));


In [ ]:
utils.embed_lecture_slides("03_NeuralNetworks/03-regression-deck.html#/design-matrix")

### Linear Model
- Polynomial model is no longer linear in the input $x^1$
- But still linear in its parameter $w$ (hence it's a linear model by definition)

### Design Matrix
- Like we did with the intercept, we include the polynomial features in the design matrix $\Phi=[x^0, x^1,\dots]$
- $\text{Polynomial Model } = \text{Linear Model } \times\text{ Polynomial Features (aka Design Matrix)}$

Note:
- in this notebook we still use $X$ to refer to $\Phi$

### Analytic Solution
- we derive the analytic solution according to the *normal equation*

In [ ]:
class LinearModel():
    def __init__(self):
        self.weights = None

    def fit(self, X, y):
        """
        Parameters:
        X : numpy array, shape (n_samples, n_features)
            The independent variables.
        y : numpy array, shape (n_samples,)
            The dependent variable (e.g., price of apartments).
        """
        self.weights = np.linalg.inv(X.T @ X) @ X.T @ y

    def predict(self, X):
        """
        Parameters:
        X : numpy array, shape (n_samples, n_features)
            The independent variables.

        Returns:
        predictions : numpy array, shape (n_samples,)
        """
        return X @ self.weights

**Task: Polynomial Features**
- Complete the function that creates polynomial features for a 1D input vector

In [ ]:
def polynomial_features(x, degree):
    """
    Parameters:
    x : numpy array, shape (n_samples,)
        The independent variable.
    degree : int
        The polynomial degree.

    Returns:
        numpy array, shape (n_samples, degree + 1)
        The polynomial features (including a column of ones).
    """
    pass  # TODO

**Task: Verification**
- For degree$=1, 2, 3$ we analytically fit a linear model to the corresponding polynomial features
- We plot the corresponding polynomial regression lines

In [ ]:
x = sizes
x_mesh = np.linspace(x.min(), x.max(), 50)
y = prices
model = LinearModel()

%matplotlib inline
plt.close()

_, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for (degree, ax) in enumerate(axes.flatten(), 1):

    try:
        X = polynomial_features(x, degree)
        X_mesh = polynomial_features(x_mesh, degree)
        assert len(X.shape) == 2 and X.shape[1] == degree + 1
        assert len(X_mesh.shape) == 2 and X_mesh.shape[1] == degree + 1
    except:
        print("Task not successful.")
        break

    model.fit(X, y)
    predictions = model.predict(X)
    predictions_mesh = model.predict(X_mesh)
    plot_1d(ax, x, y, predictions, x_mesh, predictions_mesh)
    mse = mean_squared_error(y, predictions, verbose=False)

    ax.set_xlabel("Apartment Size (sqm)")
    ax.set_ylabel("Price (EUR)")
    ax.set_title(f"Degree {degree}: MSE {mse:,.1f}")

plt.suptitle("Simple Polynomial Regression: Size vs Price")
plt.show()

## Multivariate Polynomial Regression with sklearn

**Multivariate** Polynomial? E.g. check out this $w$-parameterized ($w\in \mathbb{R}^6$) degree-$2$ polynomial $p_w$ in two variables $a, b$:
$$p_w(a, b) = \begin{bmatrix} 1 & a & b & a^2 & ab & b^2 \end{bmatrix}\begin{bmatrix} w_0 \\ w_1 \\ w_2 \\ w_3 \\ w_4 \\ w_5 \end{bmatrix}$$

**Task: Multivariate Polynomial Regression with sklearn**
- Use `sklearn` to produce multivariate polynomial features of degree $2$ in the variables `sizes`, `distances`
    - simpler than producing them manually as before
- Use a linear `sklearn` model to predict `prices` from the polynomial features

In [ ]:
# No TODOs in this cell

y = prices
degree = 2
x1 = sizes
x2 = distances
X_raw = np.column_stack((x1, x2))  # raw i.e. no polynomial features

# ugly meshgrid creation for 2D plotting
X1_mesh, X2_mesh = np.meshgrid(
    np.linspace(x1.min(), x1.max(), 50),
    np.linspace(x2.min(), x2.max(), 50)
)
X_raw_mesh = np.column_stack((X1_mesh.flatten(), X2_mesh.flatten()))

In [ ]:
from sklearn.preprocessing import PolynomialFeatures  # use this to produce polynomial features
from sklearn.linear_model import LinearRegression  # use this to solve the optimization target

model = None  # TODO: sklearn model
X = None  # TODO: polynomial features
X_mesh = None  # TODO: polynomial features for mesh (create this analogously to `X`, but instead of `X_raw` use `X_raw_mesh`)

**Task: Verification**

In [ ]:
%matplotlib ipympl
plt.close()

try:
    model.fit(X, y)
    predictions = model.predict(X)
    predictions_mesh = model.predict(X_mesh).reshape(X1_mesh.shape)
    mse = mean_squared_error(y, predictions, verbose=True)
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(X1_mesh, X2_mesh, predictions_mesh, color='orange', alpha=0.5)
    ax.scatter(X_raw[:,0], X_raw[:,1], y, color='blue')
    ax.set_xlabel("Size (sqm)")
    ax.set_ylabel("Distance to Center (km)")
    ax.set_zlabel("Price (EUR)")
    plt.title("Multiple Polynomial Regression: Size vs (Price, Distance)")
    plt.show()

except:
    print("Task not successful.")

## Synthetic Dataset

In [ ]:
utils.embed_lecture_slides("03_NeuralNetworks/03-regression-deck.html#/analytic-approach-to-function-approximation")

In [ ]:
def generate_sine_data(n_points, noise=0.2, seed=4):
    """
    Generates a synthetic dataset based on the sine function with added noise.

    Parameters:
    n_points : int
        The number of data points to generate.
    noise : optional, float
        The standard deviation of the noise to add to the sine values.
    seed : optional, int
        Enable reproducible results by controlling randomness.

    Returns:
    tuple: Two numpy arrays, x and y, where x is in the range [0, 2*pi] and
           y is the sine of x with added noise.
    """
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 2 * np.pi, n_points)
    y = np.sin(x) + rng.normal(0, noise, size=x.shape)
    return x, y

In [ ]:
x, y = generate_sine_data(10)
x_mesh = np.linspace(0, 2 * np.pi, 50)
y_mesh = np.sin(x_mesh)

%matplotlib inline
plt.close()
_, ax = plt.subplots()
ax.plot(x_mesh, y_mesh, color='red', label="Sine")
ax.scatter(x=x, y=y, color='blue', label="Noisy Observations")
plt.xlabel(None)
plt.ylabel(None)
plt.legend()
plt.title("Noisy Sine Data")
plt.show()

## Model Complexity

In [ ]:
utils.embed_lecture_slides("03_NeuralNetworks/03-regression-deck.html#/tasks-compare-different-models")

**Task: Optimal Model Complexity**
1. Find the polynomial degree $d$ for which the corresponding polynomial regression has lowest MSE on the Sine Data
2. Are you happy with this polynomial degree? Why (not)?
3. Optional: The higher the degree, the lower the MSE in theory - does this hold in practice?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

x, y = generate_sine_data(10)
x_mesh = np.linspace(0, 2 * np.pi, 50)
y_mesh = np.sin(x_mesh)

sklearn_model = LinearRegression(fit_intercept=False)
minimal_mse = None
optimal_degree = None

for degree in range(15):
    X = PolynomialFeatures(degree=degree).fit_transform(x.reshape(-1, 1))
    #  TODO ...

## Generalization Error

In [ ]:
utils.embed_lecture_slides("03_NeuralNetworks/03-regression-deck.html#/selection-of-hyperparameters/0")

In [ ]:
x_train, y_train = generate_sine_data(10)
x_test, y_test = generate_sine_data(10, seed=42)
x_mesh = np.linspace(0, 2 * np.pi, 50)
y_mesh = np.sin(x_mesh)

%matplotlib inline
plt.close()
_, ax = plt.subplots()
ax.plot(x_mesh, y_mesh, color='red', label="Sine")
ax.scatter(x_train, y_train, color='blue', label="Train Set")
ax.scatter(x_test, y_test, color='green', label="Test Set")
plt.xlabel(None)
plt.ylabel(None)
plt.legend()
plt.title("Dataset Split")
plt.show()

### Generalization Error by Model Complexity

In [ ]:
sklearn_model = LinearRegression(fit_intercept=False)

%matplotlib inline
plt.close()

minimal_mse = None
optimal_degree = None
results = []
_, axes = plt.subplots(4, 3, figsize=(15, 15), sharey=True)
for (degree, ax) in enumerate(axes.flatten(), 1):
    X_train = PolynomialFeatures(degree=degree).fit_transform(x_train.reshape(-1, 1))
    X_test = PolynomialFeatures(degree=degree).fit_transform(x_test.reshape(-1, 1))
    X_mesh = PolynomialFeatures(degree=degree).fit_transform(x_mesh.reshape(-1, 1))

    sklearn_model.fit(X_train, y_train)
    mse = mean_squared_error(y_test, sklearn_model.predict(X_test), verbose=False)
    if minimal_mse is None or mse < minimal_mse:
        minimal_mse = mse
        optimal_degree = degree
    mse_train = mean_squared_error(y_train, sklearn_model.predict(X=X_train), verbose=False)
    mse_test = mean_squared_error(y_test, sklearn_model.predict(X=X_test), verbose=False)
    results.append((degree, mse_train, mse_test))
    
    ax.set_ylim(-1.5, 1.5)
    ax.plot(x_mesh, y_mesh, color='red', label="Sine")
    ax.plot(x_mesh, sklearn_model.predict(X=X_mesh), color='blue', label="Fitted Model")
    ax.scatter(x_train, y_train, color='blue', label="Validation Set")
    ax.scatter(x_test, y_test, color='green', label="Test Set")
    ax.legend()
    ax.set_title(f"Degree {degree}: Test MSE {mse:.1e}")

print(f"Degree {optimal_degree} achieves optimal Generalization Error of {minimal_mse:.1e}")

### Generalization Error Curve

In [ ]:
degree, mse_train, mse_test = zip(*results)
mse_test_clipped = np.clip(mse_test, 0, 1)  # outliers distort the visualization
plt.plot(degree, mse_train, label='Train MSE')
plt.plot(degree, mse_test_clipped, label='Test MSE')
plt.xlabel('Polynomial Degree')
plt.ylabel('MSE')
plt.title('Train and Test MSE by Polynomial Degree')
plt.legend()
plt.show()

## Beyond Polynomial Features
- we can choose arbitrary basis functions in the design matrix $\Phi$ resp. $X$

**Task: Feature Engineering - A**
- there are 4 datasets (already split into train/test) in `./data`
    - they share the same x values
- for each dataset: find a design matrix (aka features) that enables good linear regression

In [ ]:
x_train = np.load("data/x_train.npy")
x_test = np.load("data/x_test.npy")

y1_train = np.load("data/y1_train.npy")
y1_test = np.load("data/y1_test.npy")

# y2_train = ...

model = LinearRegression(fit_intercept=True)
X_train = x_train.reshape(-1, 1) # TODO e.g. sklearn.PolynomialFeatures, e.g. np.column_stack((x_train, np.exp(x_train), np.sin(x_train), ...
X_test = x_test.reshape(-1, 1) # TODO apply same transformation as for X_train

model.fit(X_train, y1_train) # fit on train data
predictions = model.predict(X_test) # prediction on test data
mse = mean_squared_error(predictions, y1_test, verbose=False)
print(f"{mse:.2e}")
plt.scatter(x_test, predictions)
plt.scatter(x_test, y1_test)

**Task: Feature Engineering - B**
- find features that enable good linear regression on the Zebra pattern
- we interprete the Zebra pattern image as a function from pixel index to pixel color
    - we scale x and y pixel coordinate (input variable) to $[0, 1]$ each
    - we scale pixel color (output variable) to $[-1, 1]$

In [ ]:
from PIL import Image

def generate_img_dataset(n_points, seed=0):
    """
    Generates a synthetic dataset based on a 2D grayscale image.

    Parameters:
    n_points : int
        The number of data points to generate.
    seed : optional, int
        Enable reproducible results by controlling randomness.

    Returns:
    tuple:
        img, numpy array, shape (200, 200)
        X, numpy array, shape (n_points, 2)
        y, numpy array, shape (n_points,)
    """
    rng = np.random.default_rng(seed)

    size = 200
    img = Image.open("data/zebra.jpg").convert("L")  # convert to grayscale ("L" mode)
    img = img.resize((size, size), Image.Resampling.LANCZOS)
    img = 2 * (np.array(img, dtype=np.float32) / 255.0) - 1  # normalize pixel colors to [-1, 1]

    X = rng.uniform(0, 1, (n_points, 2))  # pixel coordinates for n_points samples
    coords = (img.shape[0] * X.T).astype(int)
    y = img[coords[0], coords[1]]  # pixel color for n_points samples
    
    return img, X, y

In [ ]:
%matplotlib inline
plt.close()

img, X, y = generate_img_dataset(5)
plt.imshow(img, extent=[0, 1, 0, 1], cmap='gray')
plt.scatter(X[:,1], 1 - X[:,0], c=y, vmin=-1, vmax=1, s=100, edgecolors='none')  # some coordinate transformation to fit same coordinate system as .imshow

**Task: Your input is required in the cell below**

In [ ]:
# TODO: (better) feature engineering
def add_features(X):
    X = PolynomialFeatures(degree=20).fit_transform(X)
    return X

**Task: Verification**

In [ ]:
img, X_train, y_train = generate_img_dataset(10_000)
_, X_test, y_test = generate_img_dataset(1_000, seed=42)

x1 = X_train[:,0]  # pixel x-coordinate
x2 = X_train[:,1]  # pixel y-coordinate

x1_test = X_test[:,0]
x2_test = X_test[:,1]

X_train = add_features(X_train)
sklearn_model = LinearRegression(fit_intercept=True)
sklearn_model.fit(X_train, y_train)

X_test = add_features(X_test)
preds = sklearn_model.predict(X_test)
mse = mean_squared_error(y_test, preds, verbose=False)
print(f"{mse:.2e}")

%matplotlib inline
plt.close()

fig, axes = plt.subplots(1, 2, figsize=(15, 15), sharey=True)
ax = axes.flatten()[0]

ax.imshow(img, extent=[0, 1, 0, 1], cmap='gray')
ax.set_title("Test Samples Prediction (color) on Ground Truth Background (grayscale)")

sc = ax.scatter(x2_test, 1 - x1_test, c=preds, vmin=-1, vmax=1, s=40, edgecolors='none')  # some coordinate transformation to fit .imshow
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("scaled x index")
ax.set_ylabel("scaled y index")

# colorbar for right plot
cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)


ax = axes.flatten()[1]
n_mesh = 100
x1_mesh, x2_mesh = np.meshgrid(
    np.linspace(0, 1, n_mesh),
    np.linspace(0, 1, n_mesh)
)
X_mesh = add_features(np.column_stack((x1_mesh.flatten(), x2_mesh.flatten())))
mesh_preds = sklearn_model.predict(X=X_mesh)
ax.imshow(mesh_preds.reshape((n_mesh, n_mesh)).T, vmin=-1, vmax=1, extent=[0, 1, 0, 1], cmap='gray')
ax.set_title("Mesh Prediction (grayscale)")

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("scaled x index")
ax.set_ylabel("scaled y index")

# colorbar for right plot
cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)


plt.tight_layout()
plt.show()